# Xarray in 10 Minutes

## A quick tutorial for beginners

**Goal:** Learn how to create, inspect, select, calculate, and plot labeled multidimensional data with `xarray`.

By the end, you will be able to:

1. Explain the difference between a `DataArray` and a `Dataset`.
2. Select data using coordinate labels.
3. Calculate averages across named dimensions.
4. Make a quick plot.

This notebook uses a small, self-contained weather dataset, so no download is required.

## 1. Import the packages — 1 minute

`xarray` works especially well with NumPy, pandas, Matplotlib, and NetCDF data.

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

print("xarray version:", xr.__version__)

ModuleNotFoundError: No module named 'xarray'

## 2. Create a small weather dataset — 2 minutes

An xarray object stores:

- **data variables**: the measured values, such as temperature;
- **dimensions**: the axes of the data, such as time, latitude, and longitude;
- **coordinates**: labels along those dimensions;
- **attributes**: metadata, including units and descriptions.

A `DataArray` contains one labeled variable. A `Dataset` can contain several variables that share coordinates.

In [ ]:
# Coordinate labels
times = pd.date_range("2026-09-01", periods=4, freq="6h")
latitudes = np.array([35.0, 40.0, 45.0])
longitudes = np.array([-90.0, -85.0, -80.0, -75.0])

# Make reproducible synthetic temperature data in kelvin.
rng = np.random.default_rng(seed=42)
temperature = 288 + 5 * rng.random(
    (len(times), len(latitudes), len(longitudes))
)

ds = xr.Dataset(
    data_vars={
        "temperature": (
            ("time", "latitude", "longitude"),
            temperature,
            {
                "long_name": "2 m air temperature",
                "units": "K",
            },
        )
    },
    coords={
        "time": times,
        "latitude": latitudes,
        "longitude": longitudes,
    },
    attrs={"description": "Small synthetic weather dataset"},
)

ds

### Read the dataset summary

In the output above, find:

- the three dimensions;
- their sizes;
- the `temperature` variable;
- the temperature units.

Use square brackets to extract one variable as a `DataArray`.

In [ ]:
temperature_da = ds["temperature"]

print(type(ds))
print(type(temperature_da))
print("Dimensions:", temperature_da.dims)
print("Shape:", temperature_da.shape)

temperature_da

## 3. Select data — 2 minutes

Two common selection methods are:

- `.isel()` — select by integer position;
- `.sel()` — select by coordinate label.

Using coordinate labels often makes scientific code easier to read.

In [ ]:
# First time step, selected by position
first_time = ds["temperature"].isel(time=0)

# Temperature at the grid point nearest 41°N, 86°W
near_location = ds["temperature"].sel(
    latitude=41.0,
    longitude=-86.0,
    method="nearest",
)

print("First map shape:", first_time.shape)
print("Coordinates actually selected:")
print(" latitude =", float(near_location.latitude.values))
print(" longitude =", float(near_location.longitude.values))

near_location

## 4. Calculate with named dimensions — 2 minutes

With xarray, calculations can refer to dimension names instead of axis numbers.

Below, we convert kelvin to degrees Celsius and calculate a spatial mean at each time.

In [ ]:
# Xarray preserves the coordinates during arithmetic.
ds["temperature_c"] = ds["temperature"] - 273.15
ds["temperature_c"].attrs = {
    "long_name": "2 m air temperature",
    "units": "°C",
}

# Average over latitude and longitude, leaving a time series.
regional_mean = ds["temperature_c"].mean(
    dim=["latitude", "longitude"]
)

regional_mean

## 5. Plot the data — 1 minute

Xarray uses the coordinate labels automatically when plotting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ds["temperature_c"].isel(time=0).plot(
    ax=axes[0],
    cmap="coolwarm",
    cbar_kwargs={"label": "Temperature (°C)"},
)
axes[0].set_title("Temperature at the first time")

regional_mean.plot(ax=axes[1], marker="o")
axes[1].set_title("Regional-mean temperature")
axes[1].set_ylabel("Temperature (°C)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Two-minute practice

Complete these tasks:

1. Select the temperature at the **last time step**.
2. Find the **maximum Celsius temperature** across all times and locations.
3. Calculate the mean Celsius temperature at each latitude by averaging over `time` and `longitude`.

Try the tasks before opening the solution cell.

In [3]:
# Write your answers here.

last_time = ds["temperature_c"].isel(time=-1)
maximum_temperature = ds["temperature_c"].max()
mean_by_latitude = ds["temperature_c"].mean(
    dim=["time", "longitude"]
)

last_time, maximum_temperature, mean_by_latitude

NameError: name 'ds' is not defined

In [ ]:
# Solution

last_time = ds["temperature_c"].isel(time=-1)
maximum_temperature = ds["temperature_c"].max()
mean_by_latitude = ds["temperature_c"].mean(
    dim=["time", "longitude"]
)

print("Maximum temperature:", float(maximum_temperature.values), "°C")
mean_by_latitude

## Key takeaways

- `DataArray` represents one labeled multidimensional variable.
- `Dataset` stores multiple variables that may share coordinates.
- Use `.isel()` for positions and `.sel()` for coordinate labels.
- Use named dimensions in calculations, such as `.mean(dim="time")`.
- Xarray automatically uses labels and metadata when plotting.

### Working with a real NetCDF file

```python
ds = xr.open_dataset("your_file.nc")
print(ds)
```

Always inspect a new dataset before assuming its variable names, dimensions, coordinate order, or units.